In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "AI engineers build machine learning models.",
    "A software developer writes code.",
    "Bananas are yellow."
]

embeddings = model.encode(sentences)

print(embeddings.shape)   # (3, 384)
print(embeddings[0][:5])  # first 5 values of the vector


/opt/miniconda3/envs/hf-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/hf-env/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


(3, 384)
[-0.03720323 -0.08185264  0.05348251  0.0337255   0.03474866]


In [2]:
import faiss
import numpy as np

# Suppose we have our embeddings as a numpy array
embeddings = np.array(embeddings).astype('float32')

# Create FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])  # L2 distance
index.add(embeddings)  # Add your vectors

# Perform search
query = model.encode(["Who builds machine learning models?"]).astype('float32')
D, I = index.search(query, k=2)  # top 2 most similar sentences
print(I, D)


[[0 1]] [[0.39855844 1.3559794 ]]


In [5]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

reduced = TSNE(n_components=2).fit_transform(embeddings)
plt.scatter(reduced[:,0], reduced[:,1])
plt.title("Sentence Embeddings Visualization")
plt.show()


ValueError: perplexity (30.0) must be less than n_samples (3)

In [6]:
faiss.write_index(index, "knowledge_index.faiss")

# Later load it
index = faiss.read_index("knowledge_index.faiss")
